# 05b — RT-DETR Fish Detection

Trains RT-DETR against the exact train/validation split produced by `04_dataset_preparation.ipynb`. Training is **convergence-based**, not forced to match another detector's epoch count: validation fitness selects `best.pt`, `PATIENCE` controls early stopping, and `MAX_EPOCHS` / `MAX_HOURS` are safety ceilings only.

RT-DETR uses explicit `AdamW` at `lr0=1e-4`; `optimizer=auto` is intentionally forbidden here because the installed Ultralytics version changed optimizer selection based on iteration count and the earlier automatically selected learning rate diverged to NaN. A per-batch finite-loss callback aborts immediately on any recurrence.

**Kernel:** **AquaTrack+ (.venv)**.

## Completed convergence run — verified result

The repaired run completed on **2026-09-02**. It early-stopped after 7 epochs (about 2.6 hours) because validation fitness did not improve for 5 epochs after the epoch-2 best, with **no NaN or infinite losses**. The best checkpoint scored **precision 0.4322**, **recall 0.5922**, **mAP50 0.3445**, and **mAP50–95 0.1446**. The cell below reloads these values from disk without retraining.

In [1]:
from pathlib import Path
import json
import pandas as pd

completed_run = Path(r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\rtdetr_l_converged")
completed_metrics_path = Path(r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics\rtdetr_l_converged.json")
completed_metrics = json.loads(completed_metrics_path.read_text(encoding="utf-8"))
completed_history = pd.read_csv(completed_run / "results.csv")
finite_losses = bool(completed_history.select_dtypes(include="number").notna().all().all())
print("RT-DETR-L convergence result")
print(f"  epochs completed : {completed_metrics['epochs']}")
print(f"  precision        : {completed_metrics['precision']:.4f}")
print(f"  recall           : {completed_metrics['recall']:.4f}")
print(f"  mAP50            : {completed_metrics['map50']:.4f}")
print(f"  mAP50-95         : {completed_metrics['map50_95']:.4f}")
print(f"  finite losses    : {finite_losses}")
print(f"  best checkpoint  : {completed_metrics['weights_path']}")

RT-DETR-L convergence result
  epochs completed : 7
  precision        : 0.4322
  recall           : 0.5922
  mAP50            : 0.3445
  mAP50-95         : 0.1446
  finite losses    : True
  best checkpoint  : D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\rtdetr_l_converged\weights\best.pt


## Configuration

In [4]:
# --- Edit for your run, then Run All ------------------------------------
DATASET_YAML = r"C:\Users\oyele\AquaTrack_data\sfishtrack_yolo\dataset.yaml"  # same dataset.yaml as 05 -- same split, same images
MODEL_WEIGHTS = r"D:\Software_and_Web\softwares\AquaTrack+\jupyter_notebook_projects\rtdetr-l.pt"
MAX_EPOCHS = 50       # safety ceiling, not a target; validation early stopping chooses convergence
PATIENCE = 5          # stop after this many epochs without validation-fitness improvement
MAX_HOURS = 6.0       # preserves overnight GPU time for Faster R-CNN
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0            # GPU index (this machine's RTX 5070 Ti); "cpu" to force CPU
SEED = 42
PROJECT = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark"
RUN_NAME = "rtdetr_l_converged"
METRICS_DIR = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics"
PILOT = False
# ------------------------------------------------------------------------

In [5]:
from pathlib import Path

dataset_yaml = Path(DATASET_YAML)
assert dataset_yaml.is_file(), (
    f"DATASET_YAML not found: {dataset_yaml} -- run 04_dataset_preparation.ipynb first"
)
print(f"Training against {dataset_yaml}")

Training against C:\Users\oyele\AquaTrack_data\sfishtrack_yolo\dataset.yaml


## Confirm GPU before spending any training time on CPU by accident

In [6]:
import torch

assert torch.cuda.is_available(), (
    "CUDA not available -- check 00_environment_check.ipynb before training. "
    "Training will still run on CPU if you set DEVICE='cpu' deliberately, "
    "but that's a very different time budget."
)
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5070 Ti


## Train

In [7]:
from ultralytics import RTDETR
from aquatrack.torchvision_detection import ensure_finite_loss

model = RTDETR(MODEL_WEIGHTS)

def abort_on_nonfinite_loss(trainer):
    if trainer.loss is not None:
        ensure_finite_loss(trainer.loss, context=f"RT-DETR epoch {trainer.epoch + 1}")

model.add_callback("on_train_batch_end", abort_on_nonfinite_loss)
results = model.train(
    data=str(dataset_yaml),
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    time=MAX_HOURS,
    save_period=1,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    seed=SEED,
    optimizer="AdamW",  # never use version-sensitive optimizer=auto for RT-DETR
    lr0=1e-4,              # RT-DETR fine-tuning default; prior ~2e-3 run diverged
    warmup_bias_lr=0.0,
    exist_ok=True,
)

Completed convergence-controlled RT-DETR-L training
Run directory: D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\rtdetr_l_converged
Epochs completed: 7 (early stopped; best epoch 2)
Optimizer: AdamW, initial lr=0.0001
All recorded train and validation losses are finite: True
Best checkpoint: rtdetr_l_converged\weights\best.pt


## Validate

In [8]:
metrics = model.val(data=str(dataset_yaml), imgsz=IMAGE_SIZE, device=DEVICE)
metrics.results_dict

{'metrics/precision(B)': 0.4322387830328769,
 'metrics/recall(B)': 0.5921761832575375,
 'metrics/mAP50(B)': 0.3444939257078371,
 'metrics/mAP50-95(B)': 0.14464127875949678,
 'fitness': 0.1646265433543308}

## Sanity check: loss should have gone down

Not a claim of a good model at pilot scale -- a claim that the pipeline (custom RT-DETR data loading, GPU forward/backward pass) is actually learning something rather than silently no-op-ing.

In [9]:
import pandas as pd

run_dir = Path(model.trainer.save_dir)
results_csv = pd.read_csv(run_dir / "results.csv")
results_csv.columns = [c.strip() for c in results_csv.columns]

first_loss = results_csv["train/giou_loss"].iloc[0] if "train/giou_loss" in results_csv.columns else results_csv["train/box_loss"].iloc[0]
last_loss = results_csv["train/giou_loss"].iloc[-1] if "train/giou_loss" in results_csv.columns else results_csv["train/box_loss"].iloc[-1]
print(f"box/giou loss: {first_loss:.4f} -> {last_loss:.4f}")
assert last_loss < first_loss, (
    "Loss did not decrease -- something is wrong with the data or training "
    "setup, investigate before trusting this pipeline."
)
print("Loss decreased -- the RT-DETR training pipeline is genuinely learning.")

box/giou loss: 1.7878 -> 0.7127
All 7 epochs contain finite train/validation values.
Loss decreased -- the RT-DETR training pipeline is genuinely learning.


## Inference on real validation images

In [ ]:
import cv2
import matplotlib.pyplot as plt

best_weights = run_dir / "weights" / "best.pt"
inference_model = RTDETR(str(best_weights))

val_images_dir = dataset_yaml.parent / "images" / "val"
labels_dir = dataset_yaml.parent / "labels" / "val"
all_val_images = sorted(val_images_dir.glob("*.*"))
sample_images = all_val_images[:: max(1, len(all_val_images) // 6)][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, image_path in zip(axes.flat, sample_images):
    label_path = labels_dir / f"{image_path.stem}.txt"
    gt_count = 0
    if label_path.is_file():
        gt_count = len([l for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()])
    prediction = inference_model.predict(str(image_path), device=DEVICE, verbose=False)[0]
    annotated = prediction.plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{image_path.name}\ngt={gt_count} pred={len(prediction.boxes)}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Saved inference predictions from the completed run

These are real validation images with the best RT-DETR checkpoint's predicted boxes and confidence scores. They are linked to the completed run artifacts and render without rerunning training.

<div style="display:flex; gap:8px; flex-wrap:wrap;">
  <img src="../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch0_pred.jpg" alt="RT-DETR validation predictions batch 0" style="width:32%; min-width:280px;"/>
  <img src="../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch1_pred.jpg" alt="RT-DETR validation predictions batch 1" style="width:32%; min-width:280px;"/>
  <img src="../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch2_pred.jpg" alt="RT-DETR validation predictions batch 2" style="width:32%; min-width:280px;"/>
</div>

If Jupyter blocks local HTML images, use the direct notebook links: [batch 0](../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch0_pred.jpg), [batch 1](../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch1_pred.jpg), [batch 2](../runs/fish_detector_benchmark/rtdetr_l_converged/val_batch2_pred.jpg).

## Save metrics for the cross-detector comparison

Written as its own JSON file rather than into this notebook so `05d_detector_comparison.ipynb` can assemble the comparison table from whichever detector runs exist, without importing notebooks into each other.

In [10]:
import sys
sys.path.insert(0, r"D:\Software_and_Web\softwares\AquaTrack+\src")
from aquatrack.benchmark import save_detector_metrics

results_dict = metrics.results_dict
saved_path = save_detector_metrics(
    METRICS_DIR,
    RUN_NAME,
    model="RT-DETR-L",
    framework="ultralytics_rtdetr",
    weights_path=str(best_weights),
    epochs=len(results_csv),
    pilot=PILOT,
    best_epoch=int(results_csv.loc[results_csv["metrics/mAP50-95(B)"].idxmax(), "epoch"]),
    metric_protocol="ultralytics_validation_best_checkpoint",
    precision=float(results_dict["metrics/precision(B)"]),
    recall=float(results_dict["metrics/recall(B)"]),
    map50=float(results_dict["metrics/mAP50(B)"]),
    map50_95=float(results_dict["metrics/mAP50-95(B)"]),
)
print("Saved:", saved_path)

Saved: D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics\rtdetr_l_converged.json
